<a href="https://colab.research.google.com/github/kiarashfallah/youtube-downloader/blob/youtube-downloader-gui/youtube_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tqdm
!pip install -U yt-dlp
!pip install ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.8 MB/s eta 0:00:00


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Imports
import yt_dlp
import ipywidgets as widgets
from IPython.display import display, clear_output
import os, shutil

# 3. Google Drive target folder
drive_folder = "/content/drive/MyDrive/YouTubeDownloads"
os.makedirs(drive_folder, exist_ok=True)

# 4. Get unique resolution video formats
def get_formats(url):
    ydl_opts = {'quiet': True, 'skip_download': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        formats = []
        seen_res = set()
        for f in sorted(info['formats'], key=lambda x: x.get('height') or 0, reverse=True):
            height = f.get('height')
            if f.get('vcodec') != 'none' and height and height not in seen_res:
                label = f"{height}p - {f['ext']}"
                formats.append((label, f['format_id']))
                seen_res.add(height)
            if len(formats) >= 10:
                break
        return formats, info

# 5. Widgets
url_input = widgets.Text(description='YouTube:', placeholder='Paste URL')
fetch_button = widgets.Button(description='Fetch', button_style='info')
quality_dropdown = widgets.Dropdown(description='Quality:')
download_button = widgets.Button(description='Download', button_style='success')
output = widgets.Output()

# 6. Fetch formats
def fetch_clicked(b):
    with output:
        clear_output()
        try:
            formats, info = get_formats(url_input.value.strip())
            if not formats:
                print("⚠️ No video formats found.")
                return
            quality_dropdown.options = formats
            quality_dropdown.video_info = info
            print("✅ Qualities loaded.")
        except Exception as e:
            print("❌ Error fetching formats:", e)

# 7. Download selected video-only + best audio + subtitles
from tqdm.notebook import tqdm
import logging
import sys

# Custom logger that suppresses unwanted logs
class MyLogger:
    def debug(self, msg): pass
    def warning(self, msg): pass
    def error(self, msg): print(f"❌ Error: {msg}")

def download_clicked(b):
    with output:
        clear_output()
        try:
            url = url_input.value.strip()
            selected_format = quality_dropdown.value
            title = quality_dropdown.video_info.get("title", "video").replace("/", "_")

            pbar = None

            def progress_hook(d):
                nonlocal pbar
                if d['status'] == 'downloading':
                    if not pbar:
                        total = d.get('total_bytes') or d.get('total_bytes_estimate', 0)
                        pbar = tqdm(total=total, unit='B', unit_scale=True, desc="Downloading")
                    pbar.update(d.get('downloaded_bytes', 0) - pbar.n)
                elif d['status'] == 'finished' and pbar:
                    pbar.n = pbar.total
                    pbar.refresh()
                    pbar.close()

            ydl_opts = {
                'format': f"{selected_format}+bestaudio",
                'outtmpl': f"{title}.%(ext)s",
                'writesubtitles': True,
                'writeautomaticsub': True,
                'subtitleslangs': ['en'],
                'merge_output_format': 'mp4',
                'quiet': True,
                'no_warnings': True,
                'noplaylist': True,
                'logger': MyLogger(),  # ✅ Suppresses logs
                'progress_hooks': [progress_hook]
            }

            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                file_path = ydl.prepare_filename(info)

            base = os.path.splitext(file_path)[0]
            for f in os.listdir('.'):
                if f.startswith(base):
                    shutil.move(f, os.path.join(drive_folder, f))

            print("✅ Download complete. Saved to Google Drive.")
        except Exception as e:
            print("❌ Download failed:", e)


# 8. Link and display
fetch_button.on_click(fetch_clicked)
download_button.on_click(download_clicked)
display(url_input, fetch_button, quality_dropdown, download_button, output)
